In [1]:
"""
Code used to plot the graph showcasing results.
Each experiment appends to the existing .csv file, so we average values
across all experiments before plotting
"""

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os

# Define the CSV files and their "pretty names" for the plots
# We'll automatically find all fhe_*.csv files
FILE_MAPPING = {
    'fhe_pca_results.csv': 'PCA',
    'fhe_rsvd_results.csv': 'RSVD (PCA Approx.)',
    'fhe_grp_results.csv': 'Gaussian RP',
    'fhe_srp_results.csv': 'Sparse RP',
    'fhe_jl_results.csv': 'JL (Hadamard)',
    'fhe_autoencoder_results.csv': 'Autoencoder',
}

In [2]:
def load_all_data():
    """
    Loads all .csv files, adds a 'Method' column,
    and combines them into one DataFrame.
    """
    all_dataframes = []
    
    csv_files = glob.glob('../results/fhe_*.csv')
    
    if not csv_files:
        print("Error: No 'fhe_*.csv' files found in the current directory.")
        print("Please place this script in the same folder as your CSV results.")
        return pd.DataFrame()

    print(f"Found {len(csv_files)} result files")

    for f in csv_files:
        filename = os.path.basename(f)
        
        method_name = FILE_MAPPING.get(filename)
        
        if method_name is None:
            print(f"Warning: Skipping file '{filename}' (not in FILE_MAPPING).")
            continue
            
        try:
            df = pd.read_csv(f)
            df['Method'] = method_name
            all_dataframes.append(df)
            print(f"  Loaded: {filename} (as '{method_name}')")
        except Exception as e:
            print(f"Error loading {filename}: {e}")

    if not all_dataframes:
        print("Error: No valid data was loaded.")
        return pd.DataFrame()

    # Combine all individual dataframes into one big one
    full_df = pd.concat(all_dataframes, ignore_index=True)
    
    # Convert columns to numeric, just in case
    full_df['dimension'] = pd.to_numeric(full_df['dimension'])
    full_df['avg_time_ms'] = pd.to_numeric(full_df['avg_time_ms'])
    full_df['accuracy(%)'] = pd.to_numeric(full_df['accuracy(%)'])
    
    return full_df

In [3]:
def plot_accuracy_vs_dimension(df):
    """
    Generates Plot 1: Accuracy (%) vs. Dimension (Reversed)
    """
    print("Generating 'accuracy_vs_dimension.png'...")
    plt.figure(figsize=(12, 7))
    sns.set_theme(style="whitegrid")
    
    plot = sns.lineplot(
        data=df,
        x='dimension',
        y='accuracy(%)', # or 'avg_time_ms'
        hue='Method',
        style='Method',
        markers=True,
        dashes=False,
        markersize=8,
        linewidth=2.5,
        estimator='mean',
        errorbar=('ci', 95)
    )
    
    plot.set_title('Accuracy vs. Dimensionality Reduction', fontsize=16, weight='bold')
    plot.set_xlabel('Target Embedding Dimension (Log Scale)', fontsize=12)
    
    plot.set_ylabel('Accuracy (%)', fontsize=12)
    
    # Use a log scale for the x-axis since dimensions are powers of 2
    plot.set_xscale('log', base=2)
    
    plot.invert_xaxis()
    
    # Set x-axis ticks to match the dimensions
    dims = sorted(df['dimension'].unique())
    plot.set_xticks(dims)
    plot.set_xticklabels(dims)
    
    plot.legend(title='DR Method', fontsize=11)
    plt.tight_layout()
    plt.savefig('accuracy_vs_dimension.png', dpi=300)
    plt.close()


In [4]:
def plot_time_vs_dimension(df):
    """
    Generates Plot 2: Time (ms) vs. Dimension (Reversed)
    """
    print("Generating 'time_vs_dimension.png'...")
    plt.figure(figsize=(12, 7))
    sns.set_theme(style="whitegrid")
    
    plot = sns.lineplot(
        data=df,
        x='dimension',
        y='accuracy(%)', # or 'avg_time_ms'
        hue='Method',
        style='Method',
        markers=True,
        dashes=False,
        markersize=8,
        linewidth=2.5,
        estimator='mean',  
        errorbar=('ci', 95)
    )
    
    plot.set_title('Matching Time vs. Dimensionality Reduction', fontsize=16, weight='bold')
    plot.set_xlabel('Target Embedding Dimension (Log Scale)', fontsize=12)
    
    plot.set_ylabel('Average Matching Time (ms)', fontsize=12)
    
    # Use a log scale for the x-axis
    plot.set_xscale('log', base=2)
    
    plot.invert_xaxis()
    
    # Set x-axis ticks to match the dimensions
    dims = sorted(df['dimension'].unique())
    plot.set_xticks(dims)
    plot.set_xticklabels(dims)
    
    plot.legend(title='DR Method', fontsize=11)
    plt.tight_layout()
    plt.savefig('time_vs_dimension.png')
    plt.close()


In [ ]:
def plot_tradeoff_curve(df):
    """
    Generates Bonus Plot 3: Accuracy (%) vs. Time (ms)
    (This plot does not need to be reversed)
    """
    print("Generating 'accuracy_vs_time_tradeoff.png'...")
    plt.figure(figsize=(12, 7))
    sns.set_theme(style="whitegrid")
    
    plot = sns.lineplot(
        data=df,
        x='dimension',
        y='accuracy(%)', 
        hue='Method',
        style='Method',
        markers=True,
        dashes=False,
        markersize=8,
        linewidth=2.5,
        estimator='mean',
        errorbar=('ci', 95)
    )
        
    plot.set_title('Accuracy vs. Matching Time Trade-off', fontsize=16, weight='bold')
    plot.set_xlabel('Average Matching Time (ms)', fontsize=12)
    plot.set_ylabel('Accuracy (%)', fontsize=12)
    
    # Add text labels for the dimensions
    for _, row in df.iterrows():
        # Add a small offset to avoid overlap
        plt.text(row['avg_time_ms'] + 0.5, row['accuracy(%)'] + 0.05, int(row['dimension']), fontsize=8, color='gray')

    plot.legend(title='DR Method', fontsize=11)
    plt.tight_layout()
    plt.savefig('accuracy_vs_time_tradeoff.png', dpi=300)
    plt.close()


In [9]:
def main():
    df = load_all_data()
    
    if df.empty:
        return

    plot_accuracy_vs_dimension(df)
    plot_time_vs_dimension(df)
    
    plot_tradeoff_curve(df)
    
    print("\nAll plots generated successfully!")

if __name__ == '__main__':
    main()

Found 6 result files
  Loaded: fhe_autoencoder_results.csv (as 'Autoencoder')
  Loaded: fhe_pca_results.csv (as 'PCA')
  Loaded: fhe_srp_results.csv (as 'Sparse RP')
  Loaded: fhe_jl_results.csv (as 'JL (Hadamard)')
  Loaded: fhe_rsvd_results.csv (as 'RSVD (PCA Approx.)')
  Loaded: fhe_grp_results.csv (as 'Gaussian RP')
Generating 'accuracy_vs_dimension.png'...
Generating 'time_vs_dimension.png'...
Generating 'accuracy_vs_time_tradeoff.png'...

All plots generated successfully!
